In [1]:
import os
from datasets import load_dataset
dataset = load_dataset("google/dreambooth", split="train")

output_dir = "./dreambooth_test"
os.makedirs(output_dir, exist_ok=True)


/home/marat/Desktop/CourseWork/training_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████| 6/6 [00:00<00:00, 138.33 examples/s]


In [4]:
from tqdm import tqdm
import shutil

for i, item in tqdm(enumerate(dataset)):
    image = item['image']

    # Save image
    image.save(os.path.join(output_dir, f"{i}.png"))

    # Save caption
    #with open(os.path.join(output_dir, f"{i}.txt"), "w") as f:
    #    f.write(caption)

6it [00:01,  4.43it/s]


In [ ]:
import os
from PIL import Image
import torch
import clip  
from tqdm import tqdm
import shutil

dataset_dir = "./ad_images"  
output_dir = "./bright_ads" 
device = "cuda" if torch.cuda.is_available() else "cpu"

#prompt = "minimalistic product advertisement, ad banner, clean composition, soft shadows, white background, centered object, high quality"
prompt = "a bright colorful advertisement banner, vibrant colors, high saturation, bold marketing design, eye-catching, commercial ad, strong contrast, modern advertising"
top_k = 100

model, preprocess = clip.load("ViT-B/32", device=device)

images_paths = [
    os.path.join(dataset_dir, f)
    for f in os.listdir(dataset_dir)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

scores = []

text_tokens = clip.tokenize([prompt]).to(device)



for img_path in tqdm(images_paths, desc="Scoring images"):
    try:
        image = preprocess(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)
        with torch.no_grad():
            image_features = model.encode_image(image)
            text_features = model.encode_text(text_tokens)
            image_features /= image_features.norm(dim=-1, keepdim=True)
            text_features /= text_features.norm(dim=-1, keepdim=True)
            similarity = (image_features @ text_features.T).item()
            scores.append((img_path, similarity))
    except Exception as e:
        print(f"Error with {img_path}: {e}")


scores.sort(key=lambda x: x[1], reverse=True)
top_images = scores[:top_k]

os.makedirs(output_dir, exist_ok=True)
for path, sim in top_images:
    txt_path = path[:-3] + 'txt'
    filename = os.path.basename(path)
    txt_filename = os.path.basename(txt_path)
    Image.open(path).save(os.path.join(output_dir, filename))
    shutil.copyfile(txt_path, os.path.join(output_dir, txt_filename))

print(f"Saved {len(top_images)} top images to {output_dir}")

Scoring images:   1%|          | 69/12917 [00:01<04:10, 51.32it/s]


KeyboardInterrupt: 